# Exploration, qualite et statistiques des donnees

Approche en 6 phases :
1. Audit de qualite des donnees (manquants, valeurs implausibles, doublons, coherence)
2. Analyse univariee (variables explicatives + cibles frequence/severite)
3. Analyse bivariee vs cibles (avec tests statistiques)
4. Multicolinearite / redondance (decision)
5. Coherence train/test
6. Documentation / reproductibilite

## Phase 1 — Audit de qualité des données

Objectif : identifier et documenter tout problème de qualité **avant** de produire la moindre statistique descriptive ou corrélation, pour ne pas tirer de conclusions biaisées par du bruit ou des artefacts de saisie.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

RACINE = Path.cwd().parent if Path.cwd().name == "developement" else Path.cwd()
TRAIN_PATH = RACINE / "data" / "train.csv"
TEST_PATH = RACINE / "data" / "test.csv"

train = pd.read_csv(TRAIN_PATH, sep=",")
test = pd.read_csv(TEST_PATH, sep=",")

print("Train :", train.shape)
print("Test  :", test.shape)

### 1.1 Valeurs manquantes

On distingue :
- le manquant **structurel** : logiquement absent (ex. `sex_conducteur2` quand il n'y a pas de 2e conducteur), ce n'est pas un défaut de saisie.
- le manquant **réel** : donnée attendue mais absente, qui nécessite une décision d'imputation.

In [ ]:
def rapport_manquants(df, nom):
    n = len(df)
    tab = pd.DataFrame({
        "nb_manquants": df.isna().sum(),
        "pct_manquants": (df.isna().mean() * 100).round(2),
    })
    tab = tab[tab["nb_manquants"] > 0].sort_values("nb_manquants", ascending=False)
    print(f"--- {nom} ({n} lignes) : {len(tab)} colonne(s) avec manquants ---")
    return tab

display(rapport_manquants(train, "train"))
display(rapport_manquants(test, "test"))

In [ ]:
# Vérification : les manquants sur les variables du conducteur 2 sont-ils bien
# structurels (liés à l'absence de 2e conducteur) et non du manquant réel ?
cols_conducteur2 = [c for c in train.columns if c.endswith("conducteur2") or c.endswith("permis2")]

for c in cols_conducteur2:
    manquant_hors_structurel = train.loc[train["conducteur2"] == "Yes", c].isna().sum()
    print(f"{c:22s} manquants quand conducteur2='Yes' (donc inattendu) : {manquant_hors_structurel}")

### 1.2 Valeurs implausibles / placeholders

Certaines variables numériques ne peuvent physiquement pas valoir 0 (poids, cylindrée, puissance d'un véhicule). Un 0 est probablement un code de valeur manquante déguisée, à vérifier avant toute statistique descriptive (sinon il fausse moyennes/corrélations).

In [ ]:
# Variables véhicule où 0 est physiquement impossible
vars_jamais_nulles = ["poids_vehicule", "cylindre_vehicule", "din_vehicule", "prix_vehicule", "vitesse_vehicule"]

for c in vars_jamais_nulles:
    nb_zero = (train[c] == 0).sum()
    print(f"{c:20s} valeurs à 0 : {nb_zero:5d} ({nb_zero/len(train):.2%})")

In [ ]:
# Cohérence logique entre variables liées (ex : on ne peut pas avoir son permis
# depuis plus longtemps que son âge, ni avoir obtenu le permis avant 16 ans)
incoherent_permis1 = (train["anciennete_permis1"] > train["age_conducteur1"]).sum()
incoherent_permis2 = (train["anciennete_permis2"] > train["age_conducteur2"]).sum()
age_permis_trop_jeune1 = ((train["age_conducteur1"] - train["anciennete_permis1"]) < 16).sum()

print(f"anciennete_permis1 > age_conducteur1 : {incoherent_permis1}")
print(f"anciennete_permis2 > age_conducteur2 : {incoherent_permis2}")
print(f"age au permis < 16 ans (conducteur1) : {age_permis_trop_jeune1}")

### 1.3 Doublons

Doublons de lignes complètes, et doublons sur les identifiants (qui devraient être uniques par construction).

In [ ]:
print("Lignes dupliquées (toutes colonnes) :", train.duplicated().sum())

for id_col in ["id_contrat", "id_client", "id_vehicule"]:
    if id_col in train.columns:
        print(f"Doublons sur {id_col:12s} : {train[id_col].duplicated().sum()}")

In [ ]:
# Les ID ne sont pas forcément uniques par construction métier (un client peut avoir
# plusieurs contrats) : on vérifie la cardinalité réelle plutôt que de supposer l'unicité.
for id_col in ["id_contrat", "id_client", "id_vehicule"]:
    print(f"{id_col:12s} : {train[id_col].nunique()} valeurs uniques sur {len(train)} lignes")

### 1.4 Cohérence de typage et d'encodage des catégorielles

Recherche de variantes d'écriture (espaces, casse, orthographe) qui créeraient de fausses modalités distinctes, et vérification des types de colonnes.

In [ ]:
display(train.dtypes.to_frame("dtype"))

cols_cat = train.select_dtypes(include="object").columns
for c in cols_cat:
    modalites = train[c].dropna().unique()
    # détection de variantes proches : casse ou espaces différents pour la même valeur
    normalisees = pd.Series(modalites).str.strip().str.lower()
    doublons_potentiels = normalisees[normalisees.duplicated()]
    if len(doublons_potentiels):
        print(f"{c} : variantes suspectes -> {list(doublons_potentiels)}")

print("Aucune variante suspecte détectée si rien n'est affiché ci-dessus.")

### 1.5 Application des décisions de nettoyage

Décisions validées :
- Recoder les 0 **et** valeurs négatives en NaN pour les variables véhicule physiquement non-nulles (poids, cylindrée, puissance, prix, vitesse).
- Avant d'exclure les lignes incohérentes âge/permis : vérifier si elles concentrent anormalement des sinistres (si oui, ce n'est pas du bruit pur et il faut les garder).
- Imputer `anciennete_vehicule` par la médiane.
- Supprimer les colonnes identifiants (`index`, `id_client`, `id_vehicule`, `id_contrat`).
- Conserver les NaN structurels de `sex_conducteur2`.

In [ ]:
# Recodage des 0 et valeurs négatives en NaN : un véhicule ne peut pas avoir
# un poids, une cylindrée, une puissance, un prix ou une vitesse <= 0.
vars_jamais_nulles_ou_negatives = ["poids_vehicule", "cylindre_vehicule", "din_vehicule", "prix_vehicule", "vitesse_vehicule"]

for df in (train, test):
    for c in vars_jamais_nulles_ou_negatives:
        df.loc[df[c] <= 0, c] = np.nan

print("Nouveaux NaN après recodage (train) :")
display(train[vars_jamais_nulles_ou_negatives].isna().sum())

In [ ]:
# Les lignes incohérentes âge/permis concentrent-elles plus de sinistres que la moyenne ?
# Si oui : ce n'est pas du bruit pur, on ne peut pas se permettre de les exclure (perte de signal).
train["sinistre"] = (train["nombre_sinistres"] > 0).astype(int)
taux_global = train["sinistre"].mean()

m1 = train["anciennete_permis1"] > train["age_conducteur1"]
m2 = (train["conducteur2"] == "Yes") & (train["anciennete_permis2"] > train["age_conducteur2"])
m3 = (train["age_conducteur1"] - train["anciennete_permis1"]) < 16
masque_incoherent = m1 | m2 | m3

print(f"Taux de sinistre global                         : {taux_global:.2%}\n")
for nom, m in [("permis1 > age1", m1), ("permis2 > age2 (cond2=Yes)", m2), ("permis avant 16 ans", m3)]:
    print(f"{nom:28s} n={m.sum():4d}  taux sinistre={train.loc[m, 'sinistre'].mean():.2%}")

print(f"\nUnion des 3 cas : n={masque_incoherent.sum()} ({masque_incoherent.mean():.2%} du train)")
print(f"Taux de sinistre (incohérentes) : {train.loc[masque_incoherent, 'sinistre'].mean():.2%}")
print(f"Taux de sinistre (cohérentes)    : {train.loc[~masque_incoherent, 'sinistre'].mean():.2%}")

**Résultat et décision** :

- `permis1 > age1` (n=12) et `permis avant 16 ans` (n=16) : taux de sinistre à 0% et 6.25% — volumes trop faibles pour conclure, pas d'écart flagrant.
- `permis2 > age2` quand `conducteur2='Yes'` (n=397, 0.79% du train) : taux de sinistre à **10.1%**, quasiment le double du taux global (5.83%).

➜ Ce sous-groupe n'est **pas du bruit pur** : il concentre deux fois plus de sinistres que la moyenne. Supprimer ces 397 lignes ferait perdre du signal prédictif. **Décision : on garde les lignes** (la cible `sinistre`/`montant_sinistre` reste fiable), mais on invalide uniquement les deux champs numériques suspects (`anciennete_permis1`/`anciennete_permis2`) en les passant à NaN pour les lignes concernées, plutôt que de supprimer la ligne entière.

In [ ]:
# On garde les lignes (la cible reste fiable) mais on invalide le champ numérique
# incohérent, qui sera imputé en Phase 2 au même titre qu'un manquant classique.
train.loc[m1 | m3, "anciennete_permis1"] = np.nan
train.loc[m2, "anciennete_permis2"] = np.nan

print("Lignes conservées :", len(train))
print("Nouveaux NaN -> anciennete_permis1 :", train["anciennete_permis1"].isna().sum())
print("Nouveaux NaN -> anciennete_permis2 :", train["anciennete_permis2"].isna().sum())

In [ ]:
# anciennete_vehicule : volume négligeable (1 valeur sur train), imputation médiane directe.
for df in (train, test):
    df["anciennete_vehicule"] = df["anciennete_vehicule"].fillna(df["anciennete_vehicule"].median())

print("Manquants restants anciennete_vehicule -> train:", train["anciennete_vehicule"].isna().sum(),
      "| test:", test["anciennete_vehicule"].isna().sum())

In [ ]:
# Suppression des colonnes identifiants : aucune valeur prédictive par construction.
cols_id = ["index", "id_client", "id_vehicule", "id_contrat"]
train = train.drop(columns=[c for c in cols_id if c in train.columns])
test = test.drop(columns=[c for c in cols_id if c in test.columns])

print("Train :", train.shape)
print("Test  :", test.shape)

### Synthèse Phase 1 — décisions appliquées

| Constat | Variable(s) | Décision appliquée |
|---|---|---|
| Manquant structurel | `sex_conducteur2` (66.81% NaN) | Conservé en NaN (modalité "pas de 2e conducteur") |
| Zéros/négatifs physiquement impossibles | `poids_vehicule`, `cylindre_vehicule`, `din_vehicule`, `prix_vehicule`, `vitesse_vehicule` | Recodés en NaN |
| Incohérence permis2 > âge2 (n=397, taux sinistre 10.1% vs 5.83% global) | `anciennete_permis2` | Ligne **conservée** (signal utile), champ invalidé en NaN |
| Incohérence permis1 > âge1 / permis avant 16 ans (n=12+16, volumes trop faibles) | `anciennete_permis1` | Ligne conservée, champ invalidé en NaN par cohérence de traitement |
| Manquant réel résiduel | `anciennete_vehicule` | Imputé par la médiane |
| Doublons de lignes | — | Aucun, aucune action |
| Identifiants sans valeur prédictive | `index`, `id_client`, `id_vehicule`, `id_contrat` | Colonnes supprimées |

*(Prochaine étape : Phase 2 — analyse univariée sur `train`/`test` nettoyés, y compris la distribution de `montant_sinistre`.)*

## Phase 2 — Analyse univariée

Objectif : caractériser chaque variable isolément (distribution, asymétrie, outliers) sur les données nettoyées en Phase 1, **avant** de chercher des relations avec la cible. On commence par les cibles elles-mêmes (fréquence et sévérité), car c'est le point le plus faible de l'exploration initiale.

Note : contrairement à `explore.ipynb`, on ne droppe pas a priori les colonnes `anciennete_info`, `freq_paiement`, `paiement`, `utilisation`, `code_postal`, `marque_vehicule`, `modele_vehicule` — on les profile d'abord pour juger de leur intérêt (cardinalité, variance) plutôt que de les écarter par réflexe.

### 2.1 Variables cibles — fréquence (`sinistre`)

In [ ]:
vc = train["sinistre"].value_counts()
print(vc)
print(f"\nTaux de sinistre : {train['sinistre'].mean():.2%}")
print(f"Ratio classe majoritaire / minoritaire : {vc[0] / vc[1]:.1f}:1")
print(f"Distribution de nombre_sinistres :")
print(train["nombre_sinistres"].value_counts().sort_index())

### 2.2 Variable cible — sévérité (`montant_sinistre`)

Absente de l'exploration initiale. On regarde la distribution sur les sinistres effectifs uniquement (`nombre_sinistres > 0`), avec et sans transformation log, pour juger si une régression directe (XGBoost) est adaptée ou s'il faut transformer la cible.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sin = train[train["nombre_sinistres"] > 0].copy()
montant = sin["montant_sinistre"]

print(montant.describe(percentiles=[.01, .25, .5, .75, .9, .99]).round(2))
print(f"\nSkewness : {montant.skew():.2f}")
print(f"Kurtosis : {montant.kurt():.2f}")
print(f"Valeurs <= 0 : {(montant <= 0).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(montant, bins=50, ax=axes[0]).set_title("montant_sinistre (brut)")
sns.histplot(np.log1p(montant), bins=50, ax=axes[1]).set_title("log1p(montant_sinistre)")
plt.tight_layout(); plt.show()

**Résultat** : skewness = 4.10, kurtosis = 31.66 sur `montant_sinistre` brut — distribution très asymétrique à queue lourde (n=2917 sinistres, médiane 1236€, moyenne 1772€, max 21827€, le 1% supérieur dépasse 6963€). Après `log1p` : skewness = 0.78, kurtosis = 0.34 — nettement plus proche d'une distribution symétrique.

➜ **Décision** : entraîner le modèle de sévérité sur `log1p(montant_sinistre)` plutôt que sur la valeur brute, et appliquer `expm1` à la prédiction. Une régression directe sur la valeur brute serait dominée par les quelques sinistres extrêmes (MSE écrasée par la queue de distribution).

### 2.3 Variables numériques explicatives

On classe les colonnes par dtype (en excluant les cibles), puis on calcule skewness/kurtosis et on détecte les outliers par la règle IQR (1.5×IQR), en plus du `describe()` déjà fait en Phase 1.

In [ ]:
cibles = ["nombre_sinistres", "montant_sinistre", "sinistre"]
cols_num = [c for c in train.select_dtypes(include="number").columns if c not in cibles]
cols_cat = [c for c in train.columns if c not in cols_num + cibles]

print("Numériques :", cols_num)
print("Catégorielles :", cols_cat)

In [ ]:
def profil_outliers_iqr(s):
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    bas, haut = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((s < bas) | (s > haut)).sum()

profil = pd.DataFrame({
    "skewness": train[cols_num].skew().round(2),
    "kurtosis": train[cols_num].kurt().round(2),
    "outliers_iqr": [profil_outliers_iqr(train[c].dropna()) for c in cols_num],
    "pct_outliers": [round(profil_outliers_iqr(train[c].dropna()) / train[c].notna().sum() * 100, 2) for c in cols_num],
})
display(profil.sort_values("pct_outliers", ascending=False))

**Lecture des résultats** : `bonus` (skew 3.44, 22.1% "outliers" IQR), `anciennete_permis2` (8.5%) et `anciennete_info` (6.9%) ressortent avec le plus de valeurs hors IQR. À interpréter avec prudence : la règle IQR sur-détecte mécaniquement sur une distribution asymétrique (skewness > 1) — ce n'est pas forcément des erreurs de saisie, juste une queue de distribution naturelle (ex. bonus-malus borné mais concentré près du minimum légal). Pas de suppression à ce stade ; à garder en tête pour le choix de modèle (les arbres/XGBoost sont robustes aux outliers, contrairement à une régression linéaire).

In [ ]:
n = len(cols_num); ncol = 4; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(18, 3.5 * nrow), squeeze=False)
for ax, c in zip(axes.flat, cols_num):
    sns.histplot(train[c].dropna(), bins=40, ax=ax).set_title(c)
for ax in axes.flat[n:]:
    ax.set_visible(False)
plt.tight_layout(); plt.show()

### 2.4 Variables catégorielles

On profile **toutes** les colonnes catégorielles restantes, y compris celles que `explore.ipynb` droppait sans analyse (`code_postal`, `marque_vehicule`, `modele_vehicule`, `freq_paiement`, `paiement`, `utilisation`) : cardinalité, modalité dominante, modalités rares. Une variable à cardinalité quasi-constante (ex. 99% une seule valeur) n'apporte rien ; une variable à très haute cardinalité (ex. `code_postal`) nécessite un traitement spécifique (regroupement géographique) plutôt qu'un drop réflexe.

In [ ]:
resume_cat = []
for c in cols_cat:
    vc = train[c].value_counts(dropna=False)
    resume_cat.append({
        "variable": c,
        "cardinalite": vc.size,
        "modalite_top": vc.index[0],
        "pct_top": round(vc.iloc[0] / len(train) * 100, 1),
        "modalites_rares_<30": (vc < 30).sum(),
    })
display(pd.DataFrame(resume_cat).sort_values("cardinalite", ascending=False))

**Lecture des résultats** :

- `code_postal` : 12 790 modalités pour 50 000 lignes, 12 581 rares (<30 occurrences) → essentiellement un quasi-identifiant en l'état, inutilisable tel quel. Piste : en extraire un code département (2 premiers chiffres) pour réduire la cardinalité à une variable géographique exploitable — à tester en Phase 3, pas un drop réflexe.
- `modele_vehicule` : 886 modalités, 686 rares (77%) → trop fin, fort risque de sparsité/overfitting si encodé tel quel. Candidat au drop ou à un regroupement.
- `marque_vehicule` : 89 modalités, cardinalité gérable (RENAULT 26.6% en tête) → candidate valable pour la Phase 3, avec regroupement des marques rares si besoin.
- `paiement` : 95.9% sur une seule modalité ("No") → quasi-constante, signal probablement faible, mais on ne la droppe pas avant d'avoir testé son lien avec la cible en Phase 3 (ne pas décider sur la seule distribution univariée).
- `freq_paiement`, `utilisation` : 4 modalités chacune, bien réparties → rien à signaler, candidates normales pour la suite.

### Synthèse Phase 2

| Constat | Décision pour la suite |
|---|---|
| `sinistre` déséquilibré 16.1:1 (5.83%) | À garder en tête pour la Phase 3/modélisation (class_weight, seuil de décision) — simple constat ici |
| `montant_sinistre` très asymétrique (skew 4.10 → 0.78 en log) | Entraîner le modèle de sévérité sur `log1p`, inverse en `expm1` |
| Outliers IQR élevés sur variables skewed (`bonus`, `anciennete_permis2`, `anciennete_info`) | Pas de suppression — cohérent avec un usage de modèles d'arbres (XGBoost/RandomForest), robustes aux outliers |
| `code_postal` quasi-identifiant (12790 modalités) | Ne pas utiliser tel quel ; tester un code département en Phase 3 |
| `modele_vehicule` trop fin (886 modalités) | Candidat au drop, à confirmer en Phase 3 (force de lien avec la cible) |
| `marque_vehicule`, `freq_paiement`, `utilisation` | Cardinalité raisonnable, à garder pour la Phase 3 |
| `paiement` quasi-constante (95.9%) | Ne pas dropper avant d'avoir testé le lien avec la cible en Phase 3 |

*(Prochaine étape : Phase 3 — analyse bivariée vs cibles, avec tests statistiques formels sur `train` nettoyé.)*

## Phase 3 — Analyse bivariée vs cibles

Objectif : quantifier le lien de chaque variable explicative avec les deux cibles (`sinistre` binaire, `montant_sinistre` sur les sinistres effectifs), avec des tests adaptés au type de variable et à l'asymétrie observée en Phase 2 — sans se fier aux seules p-values (n≈50000 rend presque tout "significatif" ; on regarde la taille d'effet).

- **Numérique vs `sinistre` (binaire)** : test de Mann-Whitney (rang, robuste à l'asymétrie) + corrélation point-bisériale, plutôt qu'un simple test de Student sur des distributions skewed.
- **Numérique vs `montant_sinistre`** : corrélation de Spearman (rang) sur le sous-ensemble sinistré, cohérent avec le choix `log1p` de la Phase 2.
- **Catégorielle vs `sinistre`** : V de Cramér (taille d'effet) + test du Chi², taux de sinistre par modalité.
- **Catégorielle vs `montant_sinistre`** : test de Kruskal-Wallis (rang, plusieurs groupes) sur le sous-ensemble sinistré.
- **Cas particulier `code_postal`** : test d'un regroupement en code département avant d'évaluer le lien avec la cible.

### 3.1 Variables numériques vs `sinistre` (fréquence)

In [ ]:
from scipy import stats

resultats_num_freq = []
for c in cols_num:
    mask = train[c].notna()
    g0 = train.loc[(train["sinistre"] == 0) & mask, c]
    g1 = train.loc[(train["sinistre"] == 1) & mask, c]
    _, p_mw = stats.mannwhitneyu(g1, g0, alternative="two-sided")
    r_pb, _ = stats.pointbiserialr(train.loc[mask, "sinistre"], train.loc[mask, c])
    resultats_num_freq.append({
        "variable": c,
        "mediane_sinistre0": round(g0.median(), 2),
        "mediane_sinistre1": round(g1.median(), 2),
        "r_pointbiserial": round(r_pb, 3),
        "p_value_mannwhitney": p_mw,
    })

df_num_freq = pd.DataFrame(resultats_num_freq).sort_values("r_pointbiserial", key=abs, ascending=False)
display(df_num_freq)

### 3.2 Variables numériques vs `montant_sinistre` (sévérité)

Sur le sous-ensemble sinistré uniquement (`sin`, défini en 2.2). Corrélation de Spearman entre chaque variable et `log1p(montant_sinistre)`.

In [ ]:
log_montant = np.log1p(sin["montant_sinistre"])

resultats_num_sev = []
for c in cols_num:
    mask = sin[c].notna()
    rho, p = stats.spearmanr(sin.loc[mask, c], log_montant[mask])
    resultats_num_sev.append({"variable": c, "spearman_rho": round(rho, 3), "p_value": p, "n": mask.sum()})

df_num_sev = pd.DataFrame(resultats_num_sev).sort_values("spearman_rho", key=abs, ascending=False)
display(df_num_sev)

### 3.3 Variables catégorielles vs `sinistre` (fréquence)

V de Cramér **corrigé du biais** (Bergsma, 2013) : la version brute surestime systématiquement l'association pour les variables à forte cardinalité (ex. `code_postal`, `modele_vehicule`) à n fini — sans correction on conclurait à tort qu'elles sont informatives.

In [ ]:
def cramers_v_corrige(x, y):
    tab = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(tab)[0]
    n = tab.to_numpy().sum()
    phi2 = chi2 / n
    r, k = tab.shape
    phi2_corr = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    r_corr = r - (r - 1) ** 2 / (n - 1)
    k_corr = k - (k - 1) ** 2 / (n - 1)
    denom = min(k_corr - 1, r_corr - 1)
    return np.sqrt(phi2_corr / denom) if denom > 0 else np.nan

resultats_cat_freq = []
for c in cols_cat:
    mask = train[c].notna()
    v = cramers_v_corrige(train.loc[mask, c], train.loc[mask, "sinistre"])
    resultats_cat_freq.append({"variable": c, "cramers_v_corrige": round(v, 3), "n": mask.sum()})

df_cat_freq = pd.DataFrame(resultats_cat_freq).sort_values("cramers_v_corrige", ascending=False)
display(df_cat_freq)

### 3.4 Variables catégorielles vs `montant_sinistre` (sévérité)

Test de Kruskal-Wallis (généralisation non-paramétrique du Mann-Whitney à >2 groupes) sur le sous-ensemble sinistré, uniquement pour les variables à cardinalité raisonnable (`code_postal` et `modele_vehicule` exclus ici : trop de groupes à faible effectif pour être interprétables, traités en 3.5).

In [ ]:
cols_cat_basse_card = [c for c in cols_cat if c not in ("code_postal", "modele_vehicule")]

resultats_cat_sev = []
for c in cols_cat_basse_card:
    mask = sin[c].notna()
    groupes = [log_montant[mask][sin.loc[mask, c] == v] for v in sin.loc[mask, c].unique()]
    groupes = [g for g in groupes if len(g) > 0]
    if len(groupes) < 2:
        continue
    h, p = stats.kruskal(*groupes)
    resultats_cat_sev.append({"variable": c, "kruskal_H": round(h, 2), "p_value": p, "n_groupes": len(groupes)})

df_cat_sev = pd.DataFrame(resultats_cat_sev).sort_values("kruskal_H", ascending=False)
display(df_cat_sev)

### 3.5 Cas `code_postal` : regroupement en code département

`code_postal` est un quasi-identifiant brut (Phase 2). On teste si un regroupement en département (2 premiers chiffres) réduit suffisamment la cardinalité pour redevenir exploitable, puis on mesure son lien avec `sinistre` via le même V de Cramér corrigé.

In [ ]:
train["departement"] = train["code_postal"].astype(str).str.zfill(5).str[:2]
mask = train["departement"].notna()

print("Cardinalité code_postal :", train["code_postal"].nunique())
print("Cardinalité departement :", train["departement"].nunique())

v_dept = cramers_v_corrige(train.loc[mask, "departement"], train.loc[mask, "sinistre"])
print(f"V de Cramér corrigé (departement vs sinistre) : {v_dept:.3f}")

### Synthèse Phase 3

**Numérique vs `sinistre` (fréquence)** — toutes les associations sont **faibles** (|r_pointbisérial| ≤ 0.10) :
- Le trio le plus lié : `anciennete_vehicule`, `debut_vente_vehicule`, `fin_vente_vehicule` (r ≈ -0.097, p < 1e-108) — véhicules plus récents associés à plus de sinistres (effet usage/kilométrage probable, à creuser en Phase 4 avec leur colinéarité mutuelle).
- Second groupe : `din_vehicule`, `prix_vehicule`, `vitesse_vehicule`, `poids_vehicule` (r ≈ 0.07-0.07, p < 1e-50) — véhicules plus puissants/chers associés à plus de sinistres.
- `duree_contrat` (p=0.98), `anciennete_permis1` (p=0.10), `age_conducteur1` (p=0.057) : **pas de lien détectable**.

**Numérique vs `montant_sinistre` (sévérité)** — associations encore plus faibles et majoritairement non significatives (n=2917) : seuls `anciennete_vehicule` (ρ=-0.085), `fin_vente_vehicule` (ρ=-0.074), `debut_vente_vehicule` (ρ=-0.072) et `cylindre_vehicule` (ρ=-0.043) passent p<0.05. Les variables de puissance/prix/vitesse, pourtant liées à la fréquence, **ne sont pas liées à la sévérité**.

**Catégorielle vs `sinistre`** (V de Cramér corrigé) :
- `type_contrat` (V=0.096) et `modele_vehicule` (V=0.073) sont les plus informatives.
- `code_postal` brut : **V=0.000**, confirme Phase 2 — aucun signal exploitable tel quel.
- Regroupement en `departement` (96 modalités) : V=0.036, un peu de signal récupéré mais reste faible — à évaluer en Phase 4 si le gain justifie la complexité ajoutée (sinon, abandon).

**Catégorielle vs `montant_sinistre`** (Kruskal-Wallis) :
- `type_contrat` seul ressort nettement (H=50.99, p<1e-10).
- `marque_vehicule` (46 groupes testés) **non significatif** (p=0.93) malgré sa cardinalité élevée → confirme le risque de sur-ajustement signalé en Phase 2 sans contrepartie de signal sur la sévérité.
- Toutes les autres catégorielles basse-cardinalité : non significatives sur la sévérité.

| Variable | Lien fréquence | Lien sévérité | Décision pour Phase 4 |
|---|---|---|---|
| `anciennete_vehicule`, `debut_vente_vehicule`, `fin_vente_vehicule` | Oui (r≈0.10) | Oui (ρ≈0.08) | Fortement corrélées entre elles (à vérifier Phase 4) — garder un sous-ensemble |
| `din_vehicule`, `prix_vehicule`, `vitesse_vehicule`, `poids_vehicule`, `cylindre_vehicule` | Oui (r≈0.07) | Faible/nul | Cluster "puissance" probable, à tester pour redondance en Phase 4 |
| `type_contrat` | Oui (V=0.096) | Oui (H=50.99) | Variable la plus informative des deux côtés — à garder |
| `modele_vehicule` | Oui (V=0.073) | Non | Signal fréquence réel mais cardinalité (886) à gérer (regroupement ou encodage spécifique) |
| `marque_vehicule` | Faible (V=0.019) | Non | Signal faible malgré la cardinalité — candidat au drop |
| `code_postal` | Nul (V=0.000) | — | Pas utilisable tel quel ; `departement` apporte un signal marginal (V=0.036), à confirmer utilité réelle |
| `duree_contrat`, `age_conducteur1`, `anciennete_permis1`, `paiement`, `freq_paiement`, `sex_conducteur1/2` | Non significatif | Non testé/non significatif | Candidats au drop, à revalider en Phase 4 (pas de décision définitive sur un seul test univarié) |

*(Prochaine étape : Phase 4 — multicolinéarité/redondance, notamment sur le cluster `anciennete_vehicule`/`debut_vente_vehicule`/`fin_vente_vehicule` et le cluster "puissance", avant de figer la liste finale de variables.)*

## Phase 4 — Associations entre variables explicatives et implications pour le choix de modèle

Objectif : la Phase 3 a mesuré le lien de chaque variable avec les cibles. Cette phase mesure le lien **entre variables explicatives elles-mêmes** (redondance/multicolinéarité), avec 8 métriques/tests complémentaires, pour répondre à une question concrète de modélisation : *quelles variables sont interchangeables, et cela change-t-il selon le modèle choisi ?*

| # | Mesure | Couple de types | Ce qu'elle capte |
|---|---|---|---|
| 1 | Pearson r | numérique-numérique | association **linéaire** |
| 2 | Spearman ρ | numérique-numérique | association **monotone** (robuste à l'asymétrie — cf. Phase 2) |
| 3 | VIF (Variance Inflation Factor) | numérique (multivarié) | colinéarité **diagnostique directe pour régression/GLM** |
| 4 | Nombre de conditionnement | numérique (matrice) | sévérité globale de la colinéarité |
| 5 | V de Cramér corrigé | catégorielle-catégorielle | association, sans le biais de cardinalité (cf. Phase 3) |
| 6 | ω² (omega carré, eta corrigé) | catégorielle-numérique | variance expliquée, corrigée du biais d'échantillon |
| 7 | Information mutuelle | numérique-numérique | association **non linéaire**, pertinente pour les modèles d'arbres |
| 8 | Clustering hiérarchique (dendrogramme) | toutes les numériques | regroupe les variables redondantes en blocs actionnables |

**Pourquoi ça compte pour le choix de modèle** : un Random Forest/XGBoost tolère la redondance (il répartit l'importance entre variables corrélées, comme vu dans `proba_prediction.ipynb` avec la permutation groupée). Une régression linéaire/logistique ou un GLM, en revanche, devient instable (coefficients qui explosent, signes contre-intuitifs) si deux variables très corrélées sont incluses ensemble — le VIF et le nombre de conditionnement servent précisément à détecter ce risque avant un éventuel usage de modèles linéaires.

### 4.1 et 4.2 — Pearson r vs Spearman ρ (numérique-numérique)

On calcule les deux pour la même raison qu'en Phase 2/3 : plusieurs variables sont asymétriques (`bonus`, `anciennete_info`, `prix_vehicule`...), Pearson peut sous-estimer une association monotone non linéaire. Un écart important entre les deux matrices sur une paire signale une relation non linéaire que Pearson seul aurait manquée.

In [ ]:
corr_pearson = train[cols_num].corr(method="pearson")
corr_spearman = train[cols_num].corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
sns.heatmap(corr_pearson, cmap="coolwarm", center=0, annot=False, ax=axes[0])
axes[0].set_title("Pearson r")
sns.heatmap(corr_spearman, cmap="coolwarm", center=0, annot=False, ax=axes[1])
axes[1].set_title("Spearman ρ")
plt.tight_layout(); plt.show()

ecart_arr = (corr_spearman - corr_pearson).abs().to_numpy().copy()
np.fill_diagonal(ecart_arr, 0)
ecart = pd.DataFrame(ecart_arr, index=cols_num, columns=cols_num)
paires_ecart = ecart.unstack().sort_values(ascending=False)
paires_ecart = paires_ecart[paires_ecart.index.get_level_values(0) < paires_ecart.index.get_level_values(1)]
print("Paires où |Spearman - Pearson| est le plus grand :")
print(paires_ecart.head(5).round(3))

print("\nPaires |Spearman| > 0.5 (hors diagonale) :")
paires_fortes = corr_spearman.abs().unstack().sort_values(ascending=False)
paires_fortes = paires_fortes[paires_fortes.index.get_level_values(0) < paires_fortes.index.get_level_values(1)]
print(paires_fortes[paires_fortes > 0.5].round(3))

**Interprétation (4.1/4.2) — comment lire ces chiffres :** les écarts |Spearman − Pearson| restent faibles (max 0,123 pour `poids_vehicule`/`vitesse_vehicule`) → les relations entre variables numériques sont essentiellement **monotones**, pas de non-linéarité cachée qui échapperait à Pearson. En revanche plusieurs paires affichent un Spearman très élevé : `age_conducteur2`/`anciennete_permis2` = **0,997** (quasi-identiques — logique, l'âge et l'ancienneté de permis du 2ᵉ conducteur évoluent presque toujours ensemble), `anciennete_vehicule`/`debut_vente_vehicule` = 0,961, `debut_vente_vehicule`/`fin_vente_vehicule` = 0,949, `anciennete_vehicule`/`fin_vente_vehicule` = 0,942 (cluster « âge du véhicule »), `age_conducteur1`/`anciennete_permis1` = 0,925, puis le bloc « puissance/gabarit » (`din_vehicule`, `prix_vehicule`, `poids_vehicule`, `vitesse_vehicule`, `cylindre_vehicule`) entre 0,56 et 0,88.

**Choix d'interprétation retenu :** on ne traite pas un |ρ| élevé comme « variable à supprimer » automatiquement — c'est un **signal de redondance à confirmer** par VIF/ω²/clustering avant toute décision, et la décision elle-même dépend du modèle visé (cf. tableau d'introduction de la Phase 4).

### 4.3 — VIF (Variance Inflation Factor)

Mesure directement actionnable pour un modèle linéaire/GLM : pour chaque variable, on régresse cette variable sur **toutes les autres** et on regarde le R² obtenu — VIF = 1/(1-R²). VIF > 5 : colinéarité préoccupante ; VIF > 10 : sévère. (Calcul manuel via `sklearn.LinearRegression`, `statsmodels` n'étant pas installé dans l'environnement — évite d'ajouter une dépendance non nécessaire.)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer

X_num = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(train[cols_num]), columns=cols_num)

def vif(X, colonne):
    autres = [c for c in X.columns if c != colonne]
    r2 = LinearRegression().fit(X[autres], X[colonne]).score(X[autres], X[colonne])
    return 1 / (1 - r2) if r2 < 1 else np.inf

vif_tab = pd.DataFrame({
    "variable": cols_num,
    "VIF": [vif(X_num, c) for c in cols_num],
}).sort_values("VIF", ascending=False)
display(vif_tab.round(2))

**Interprétation (4.3) — comment lire ces chiffres :** règle usuelle : VIF > 10 signale une redondance qui rend les coefficients d'un modèle linéaire instables (variance inflée, signes qui changent selon l'échantillon). Six variables dépassent ce seuil : `debut_vente_vehicule` (16,3), `anciennete_vehicule` (15,9), `anciennete_permis2` (13,2), `age_conducteur2` (13,1), `fin_vente_vehicule` (12,7), `din_vehicule` (10,3) — exactement les deux clusters déjà repérés en 4.1/4.2. À l'inverse `bonus`, `anciennete_info`, `duree_contrat` restent proches de 1,2-1,3 : information quasi indépendante du reste.

**Choix d'interprétation retenu :** le VIF est lu comme un diagnostic **spécifique aux modèles linéaires/GLM** — il ne remet rien en cause pour un arbre ou une forêt (qui split une variable à la fois et tolère la redondance, comme déjà montré par la permutation importance groupée de `proba_prediction.ipynb`). Pour un GLM, il sert à décider *combien* de variables garder par cluster (en général une seule), pas forcément *laquelle* — ce choix final viendra du dendrogramme (4.8) croisé avec l'interprétabilité métier.

### 4.4 — Nombre de conditionnement de la matrice de corrélation

Diagnostic global (complémentaire au VIF variable par variable) : le rapport entre la plus grande et la plus petite valeur propre de la matrice de corrélation. Un nombre de conditionnement élevé (règle usuelle : >30) signale une colinéarité multivariée, même quand aucune paire de VIF individuels ne dépasse le seuil.

In [ ]:
valeurs_propres = np.linalg.eigvalsh(corr_pearson.values)
cond = valeurs_propres.max() / valeurs_propres.min()
print(f"Valeurs propres : min={valeurs_propres.min():.3f}, max={valeurs_propres.max():.3f}")
print(f"Nombre de conditionnement : {cond:.1f}")

**Interprétation (4.4) — comment lire ces chiffres :** valeur propre minimale = 0,036, maximale = 4,679, nombre de conditionnement = **130,9**. Repère usuel : au-delà de 30 la matrice est jugée mal conditionnée pour une inversion numérique (régression linéaire), au-delà de 100 la situation est sévère. 130,9 confirme que la redondance n'est pas isolée à une ou deux paires mais **structurelle à l'échelle de la matrice entière** — cohérent avec les 6 variables à VIF > 10 trouvées en 4.3.

**Choix d'interprétation retenu :** ce diagnostic est global et ne dit pas *quoi* retirer, seulement *qu'il faut* retirer quelque chose avant d'ajuster un modèle linéaire/GLM sur l'ensemble des 15 variables numériques telles quelles. Il confirme la nécessité des sections 4.5 à 4.8 pour identifier précisément les blocs responsables.

### 4.5 — V de Cramér corrigé (catégorielle-catégorielle)

Même fonction `cramers_v_corrige` que la Phase 3 (section 3.3), appliquée cette fois entre variables explicatives catégorielles entre elles plutôt que vs la cible.

In [ ]:
from itertools import combinations

paires_cat = list(combinations(cols_cat, 2))
resultats_cramer_cat = []
for a, b in paires_cat:
    mask = train[a].notna() & train[b].notna()
    v = cramers_v_corrige(train.loc[mask, a], train.loc[mask, b])
    resultats_cramer_cat.append({"var_1": a, "var_2": b, "cramer_v_corrige": v})

df_cramer_cat = pd.DataFrame(resultats_cramer_cat).sort_values("cramer_v_corrige", ascending=False)
print("Paires avec V de Cramér corrigé > 0.3 (association notable) :")
display(df_cramer_cat[df_cramer_cat["cramer_v_corrige"] > 0.3].round(3))
print("\nTableau complet :")
display(df_cramer_cat.round(3))

**Interprétation (4.5) — comment lire ces chiffres :** `marque_vehicule`/`modele_vehicule` = **0,944**, `modele_vehicule`/`type_vehicule` = 0,791, `essence_vehicule`/`modele_vehicule` = 0,745, `sex_conducteur1`/`sex_conducteur2` = 0,723, `type_contrat`/`modele_vehicule` = 0,366. Les trois premières paires ne sont pas une coïncidence statistique mais une **hiérarchie logique** : un modèle de véhicule détermine quasi-entièrement sa marque, son type et son carburant usuel — ce n'est pas une redondance « à arbitrer », c'est une relation d'inclusion (modèle ⊂ marque).

**Choix d'interprétation retenu :** on distingue cette redondance **déterministe/hiérarchique** (modèle → marque/type/essence) d'une redondance **statistique** entre variables indépendantes a priori (ex. `type_contrat`/`modele_vehicule` = 0,366, plus faible et non déterministe). Pour un modèle linéaire, la conséquence pratique est de ne garder que la variable la plus fine de la hiérarchie (`modele_vehicule`) si le budget de modalités le permet, sinon la plus grossière (`marque_vehicule`) — jamais les deux simultanément, car l'une est presque entièrement une fonction de l'autre.

### 4.6 — ω² corrigé (omega carré, catégorielle-numérique)

**Pourquoi cette métrique et pas une simple ANOVA / η² ?** Le η² (eta carré, part de variance expliquée par les groupes) est biaisé à la hausse quand le nombre de groupes est élevé par rapport à la taille de l'échantillon — exactement le problème déjà rencontré avec le V de Cramér non corrigé en Phase 3. Le ω² applique la même logique de correction (il retire l'espérance du biais sous H0) pour rester comparable aux autres métriques corrigées de ce notebook. On l'interprète comme *« quelle part de la variance de la variable numérique est expliquée par l'appartenance aux groupes de la variable catégorielle, une fois le biais d'échantillonnage retiré »* — une association proche de 0 signifie que les deux variables apportent une information indépendante.

In [ ]:
def omega2_corrige(cat, num):
    df = pd.DataFrame({"cat": cat, "num": num}).dropna()
    k = df["cat"].nunique()
    n = len(df)
    if k < 2 or n <= k:
        return np.nan
    grand_mean = df["num"].mean()
    groupes = df.groupby("cat")["num"]
    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for _, g in groupes)
    ss_total = ((df["num"] - grand_mean) ** 2).sum()
    ss_within = ss_total - ss_between
    ms_within = ss_within / (n - k)
    if ss_total + ms_within == 0:
        return np.nan
    return max(0, (ss_between - (k - 1) * ms_within) / (ss_total + ms_within))

resultats_omega2 = []
for c in cols_cat:
    for v in cols_num:
        resultats_omega2.append({
            "cat": c, "num": v,
            "omega2_corrige": omega2_corrige(train[c], train[v]),
        })

df_omega2 = pd.DataFrame(resultats_omega2)
mat_omega2 = df_omega2.pivot(index="cat", columns="num", values="omega2_corrige")

plt.figure(figsize=(14, 6))
sns.heatmap(mat_omega2, cmap="viridis", annot=False)
plt.title("ω² corrigé — catégorielle (lignes) vs numérique (colonnes)")
plt.tight_layout()
plt.show()

print("Paires avec ω² corrigé > 0.1 :")
display(df_omega2[df_omega2["omega2_corrige"] > 0.1].sort_values("omega2_corrige", ascending=False).round(3))

**Interprétation (4.6) — comment lire ces chiffres :** `modele_vehicule` explique 65 % à 85 % de la variance de tout le bloc « puissance/gabarit » (`poids_vehicule` 0,851, `prix_vehicule` 0,850, `cylindre_vehicule` 0,802, `vitesse_vehicule` 0,786, `din_vehicule` 0,757) et du bloc « âge du véhicule » (0,65-0,71) — logique, ces caractéristiques physiques sont presque une fonction du modèle. `conducteur2` explique 0,848 de `age_conducteur2` et 0,682 de `anciennete_permis2` : ceci n'est **pas une association comportementale** mais un artefact de codage déjà identifié en Phase 1/2 — quand `conducteur2 = "No"`, ces champs sont codés à 0/manquant par construction. Enfin `utilisation` (0,498 sur `age_conducteur1`) et `type_contrat` (0,40-0,43 sur le cluster âge-véhicule) révèlent des associations métier plus substantielles (ex. usage pro vs perso corrélé à l'âge du conducteur).

**Choix d'interprétation retenu :** on sépare trois natures d'association que le chiffre brut ne distingue pas — (1) **déterministe** (modèle → caractéristiques physiques, déjà vu en 4.5), (2) **artefact de codage structurel** (`conducteur2` → champs du 2ᵉ conducteur, à traiter par un indicateur de présence plutôt que par suppression de variable), (3) **association métier réelle** (`utilisation`/`type_contrat` vs profil démographique). Seule la 3ᵉ catégorie justifie un arbitrage classique « quelle variable garder ».

### 4.7 — Information mutuelle (numérique-numérique)

**Pourquoi une 7ᵉ métrique alors qu'on a déjà Pearson et Spearman ?** Pearson capture le linéaire, Spearman capture le monotone — mais une relation peut être non-monotone (ex. en U, ou en cloche) et rester invisible aux deux. L'information mutuelle (IM) ne fait aucune hypothèse de forme : elle mesure la réduction d'incertitude sur une variable une fois l'autre connue, quelle que soit la forme de la dépendance. On l'interprète ici comme un **filet de sécurité** : si une paire a une IM élevée mais des corrélations Pearson/Spearman faibles, cela signale une redondance non-linéaire qu'on aurait manquée autrement — information actionnable pour un modèle linéaire (qui ne la capterait pas sans transformation) mais pas pour un arbre (qui la capte nativement).

In [ ]:
from sklearn.feature_selection import mutual_info_regression

mat_mi = pd.DataFrame(np.zeros((len(cols_num), len(cols_num))), index=cols_num, columns=cols_num)
for cible in cols_num:
    autres = [c for c in cols_num if c != cible]
    mi = mutual_info_regression(X_num[autres], X_num[cible], random_state=0)
    for c, val in zip(autres, mi):
        mat_mi.loc[cible, c] = val

# Symétrisation (MI(x->y) != MI(y->x) en estimation, on moyenne les deux sens)
mat_mi_sym = (mat_mi + mat_mi.T) / 2

plt.figure(figsize=(10, 8))
sns.heatmap(mat_mi_sym, cmap="magma", annot=False)
plt.title("Information mutuelle (symétrisée) entre variables numériques")
plt.tight_layout()
plt.show()

paires_mi = mat_mi_sym.unstack().sort_values(ascending=False)
paires_mi = paires_mi[paires_mi.index.get_level_values(0) < paires_mi.index.get_level_values(1)]
print("Top 10 paires par information mutuelle :")
print(paires_mi.head(10).round(3))

**Interprétation (4.7) — comment lire ces chiffres :** le classement par information mutuelle (`poids_vehicule`/`prix_vehicule` en tête, puis `cylindre_vehicule`/`prix_vehicule`, `prix_vehicule`/`vitesse_vehicule`, `din_vehicule`/`prix_vehicule`...) reproduit presque exactement le classement Spearman de 4.1/4.2 — le même bloc « puissance/gabarit » domine, et les variables de dates de vente (`debut_vente_vehicule`, `fin_vente_vehicule`) arrivent ensuite avec `prix_vehicule`.

**Choix d'interprétation retenu :** l'IM ne révèle **aucune relation non-linéaire cachée** supplémentaire par rapport à Pearson/Spearman — c'est un résultat négatif mais utile : il **valide** que les diagnostics linéaires (VIF, Pearson) suffisent ici et qu'on n'a pas besoin de transformations non-linéaires pour capter ces redondances dans un modèle linéaire. Cette métrique joue donc un rôle de garde-fou qui confirme les 6 diagnostics précédents plutôt que d'ajouter une nouvelle paire à traiter.

### 4.8 — Clustering hiérarchique (dendrogramme)

**Pourquoi un 8ᵉ outil alors qu'on a déjà 7 métriques de paires ?** Les 7 métriques précédentes donnent des scores **par paire** — elles ne disent pas comment regrouper les variables en **blocs cohérents** quand la redondance est partagée entre 3, 4 ou 5 variables à la fois (ce qui est le cas ici, cf. le cluster de vieillissement du véhicule pressenti en Phase 3). Le clustering hiérarchique sur la distance `1 - |Spearman|` répond à cette question : il agrège automatiquement les variables qui se ressemblent et permet de lire sur le dendrogramme, à une hauteur de coupe donnée, quelles variables sont interchangeables. C'est l'outil le plus directement actionnable pour la décision finale : *pour un modèle linéaire, ne garder qu'une variable représentative par cluster*.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

dist_arr = (1 - corr_spearman.abs()).to_numpy().copy()
np.fill_diagonal(dist_arr, 0)
dist_condensee = squareform(dist_arr, checks=False)
Z = linkage(dist_condensee, method="average")

plt.figure(figsize=(12, 6))
dendrogram(Z, labels=cols_num, leaf_rotation=90)
plt.axhline(0.5, color="red", linestyle="--", label="seuil de coupe (dist=0.5, soit |ρ|=0.5)")
plt.title("Clustering hiérarchique des variables numériques (distance = 1 - |Spearman|)")
plt.legend()
plt.tight_layout()
plt.show()

clusters = fcluster(Z, t=0.5, criterion="distance")
df_clusters = pd.DataFrame({"variable": cols_num, "cluster": clusters}).sort_values("cluster")
display(df_clusters)

**Interprétation (4.8) — comment lire ces chiffres :** à une coupe de distance 0,5 (soit |ρ| > 0,5), le dendrogramme regroupe les 15 variables numériques en **4 blocs + 3 variables isolées** :
- bloc « conducteur 1 » : `age_conducteur1`, `anciennete_permis1`
- bloc « conducteur 2 » : `age_conducteur2`, `anciennete_permis2`
- bloc « puissance/gabarit » (5 variables) : `din_vehicule`, `cylindre_vehicule`, `poids_vehicule`, `vitesse_vehicule`, `prix_vehicule`
- bloc « âge du véhicule » (3 variables) : `anciennete_vehicule`, `debut_vente_vehicule`, `fin_vente_vehicule`
- isolées (information quasi indépendante) : `bonus`, `duree_contrat`, `anciennete_info`

**Choix d'interprétation retenu :** ce découpage est la **synthèse opérationnelle** des 7 métriques précédentes — il ne remplace pas VIF/Cramér/ω² (qui disent *pourquoi* un bloc est redondant) mais répond à la question *combien de variables indépendantes y a-t-il vraiment* : 15 variables numériques brutes ≈ **7 sources d'information indépendantes** une fois les blocs fusionnés.

### Synthèse Phase 4 — décisions de modélisation

**Pour un arbre / une forêt / du gradient boosting (RandomForest, XGBoost) :** aucune suppression de variable n'est nécessaire. Ces modèles ne sont pas déstabilisés par la redondance (ils splittent une variable à la fois), ce qui est cohérent avec la permutation importance **groupée** de `proba_prediction.ipynb` : le bloc « âge du véhicule » reste le signal dominant (perte AUC 0,050) même en présence de ses 2 variables corrélées à 0,94-0,96.

**Pour un modèle linéaire / GLM, un représentant par bloc (cf. 4.8) :**

| Bloc | Variables redondantes | Représentant recommandé | Justification |
|---|---|---|---|
| Conducteur 1 | `age_conducteur1`, `anciennete_permis1` (ρ=0,925, VIF 6,9-6,9) | `age_conducteur1` | Facteur de risque classique, directement interprétable |
| Conducteur 2 | `age_conducteur2`, `anciennete_permis2` (ρ=0,997, VIF 13,1-13,2) | `age_conducteur2` + indicateur `conducteur2` présent | ω²=0,848 avec `conducteur2` = artefact de codage structurel, pas une vraie covariation à arbitrer seule |
| Âge véhicule | `anciennete_vehicule`, `debut_vente_vehicule`, `fin_vente_vehicule` (ρ=0,94-0,96, VIF 12,7-16,3) | `anciennete_vehicule` | Variable la plus interprétable métier, déjà utilisée comme telle en Phase 3 |
| Puissance/gabarit | `din_vehicule`, `cylindre_vehicule`, `poids_vehicule`, `vitesse_vehicule`, `prix_vehicule` (ρ jusqu'à 0,88, VIF jusqu'à 10,3) | `din_vehicule` (puissance) **ou** `prix_vehicule` selon l'angle (risque physique vs valeur à assurer) | Les deux sont défendables ; à trancher avec l'équipe métier plutôt que statistiquement — l'IM (4.7) ne permet pas de préférer l'un à l'autre |
| Catégorielles véhicule | `marque_vehicule`, `modele_vehicule`, `type_vehicule`, `essence_vehicule` (Cramér jusqu'à 0,944, hiérarchie déterministe) | `modele_vehicule` si budget de modalités suffisant, sinon `marque_vehicule` | Relation d'inclusion, pas d'arbitrage statistique à faire |
| Sexe conducteurs | `sex_conducteur1`, `sex_conducteur2` (Cramér 0,723) | `sex_conducteur1` | Moins de valeurs manquantes (défini même si `conducteur2`="No") |
| Indépendantes | `bonus`, `duree_contrat`, `anciennete_info` | toutes conservées | VIF 1,2-1,3, aucune redondance détectée sur les 8 métriques |

➜ Après réduction, un modèle linéaire tournerait sur **~7-8 variables numériques indépendantes** (au lieu de 15) + les variables catégorielles dédupliquées — un nombre de conditionnement à recalculer sur ce sous-ensemble serait l'étape de vérification naturelle avant tout ajustement GLM.